In [ ]:
import re
import os
import json


In [ ]:
project_dir_new = r"C:\Repo\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel" 
project_dir_old = r"C:\Repo\EnterpriseCloud\astro_datautility\dags\dbt\dbt_project" 
with open(os.path.join(project_dir_old,"target\\manifest.json"), 'r') as f:
    manifest = json.load(f)

In [ ]:
def replace_where_condition(sql_text, new_where):
    pattern = r"(FROM\s+\{\{\s*source\([^\}]+\)\s*\}}\s*)WHERE\s+[^\n]+"
    updated_sql = re.sub(pattern, r"\1" + new_where, sql_text, flags=re.IGNORECASE)
    return updated_sql

In [ ]:
for key,value in manifest['nodes'].items():
    if "ODS__SAPAPI__" in key and value["config"]['incremental_strategy'] == 'insert_overwrite':
        alias = value["config"]['alias']
        pattern = r"FROM\s+\{\{\s*source\('SAPH'"
        result = re.search(pattern, value['raw_code'],flags=re.DOTALL |  re.IGNORECASE)
        match_src = re.search(r"\{\{\s+source\('SAPH','(.+?)'\)\s*\}\}", value['raw_code'],re.IGNORECASE)    
        #match = re.search(r"FROM\s+\{\{\s*ref\('(.+?)'\)\s*\}\}", content,re.IGNORECASE)           
        if match_src:
            src_name = match_src.group(1)
            print(src_name)   
        if result:
            new_where = f"""WHERE _FILEDATE::STRING = (
        SELECT MAX(RUN_TIME)::STRING
        FROM {{{{ source('SAPH', 'CONTROLTABLE') }}}}
        WHERE TABLE_NAME = '{ src_name }'
        )"""
            updated_sql = replace_where_condition(value['raw_code'], new_where)
            with open(os.path.join(project_dir_old,value['original_file_path']),'w') as f:
                #f.write(updated_sql)
                f.write("\n".join(updated_sql.splitlines()))
            